In [1]:
# ==========================
# === CHANGE EPISODE PIPELINE (STRICT DEFAULT-FIRST-PARENT) ===
# Outputs:
#   1) List_Boundary_Commit_Events.csv   (boundary commits for episodes)
#   2) List_Change_Episodes.csv          (episode timeline feed)
#   3) Repo_level_episodes.csv           (repo-level first + current status)
#
# STRICTLY uses DEFAULT BRANCH MAINLINE (default_first_parent preferred).
# Skips repo-wide timelines entirely.
# ==========================

from __future__ import annotations
import json
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Set
from collections import defaultdict

import pandas as pd

# ---- Paths ----
WORK_ROOT   = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2")
OUTPUT_MINE = WORK_ROOT / "Mine_Full"

MAIN_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv")

OBS_OUTPUT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations")
OBS_OUTPUT.mkdir(parents=True, exist_ok=True)

# ---- Settings ----
CUTOFF_ISO   = "2025-08-10 23:59:59 +0000"
PERSIST_DAYS = 14.0
CANONICAL_STYLES: Set[str] = {"Emu_Community", "Emu_Custom", "GMD", "ThirdParty"}


# --------------------------
# Helpers
# --------------------------
def parse_iso(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    s = s.strip()
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    # normalize +0000 -> +00:00
    if len(s) >= 5 and (s[-5] in ["+", "-"]) and s[-3] != ":":
        s = s[:-2] + ":" + s[-2:]
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        return None

CUTOFF_DT = parse_iso(CUTOFF_ISO)
assert CUTOFF_DT is not None, f"Bad CUTOFF_ISO: {CUTOFF_ISO}"

def days_between(a: datetime, b: datetime) -> float:
    return (b - a).total_seconds() / 86400.0

def normalize_styles(style_list: List[str]) -> Set[str]:
    return {s for s in (style_list or []) if s in CANONICAL_STYLES}

def label_from_set(s: Set[str]) -> str:
    if not s:
        return "None"
    if len(s) >= 2:
        return "Mixed"
    if "Emu_Community" in s:
        return "Community"
    if "Emu_Custom" in s:
        return "Custom"
    if "GMD" in s:
        return "GMD"
    if "ThirdParty" in s:
        return "ThirdParty"
    return "None"

def styleset_to_str(styleset: Set[str]) -> str:
    return "+".join(sorted(styleset)) if styleset else ""

def read_repo_jsons(output_dir: Path) -> List[Path]:
    # Reads all emulator timeline JSONs; we will FILTER/SELECT strictly later.
    return sorted(output_dir.glob("*.emulator_timeline*.json"))

def repo_from_filename(jf: Path) -> str:
    # Extract "Owner__Repo" from: Owner__Repo.emulator_timeline.default_refs.json
    name = jf.name
    if ".emulator_timeline" in name:
        return name.split(".emulator_timeline", 1)[0]
    return jf.stem

def is_default_scoped(jf: Path, rec: Dict) -> bool:
    scope = str(rec.get("timeline_scope") or "").lower()
    nm = jf.name.lower()
    return ("default_first_parent" in scope) or ("default_refs" in scope) or ("default_refs" in nm) or ("default_first_parent" in nm)

def default_scope_score(jf: Path, rec: Dict) -> int:
    """
    Higher = better. We prefer default_first_parent over default_refs.
    """
    scope = str(rec.get("timeline_scope") or "").strip().lower()
    nm = jf.name.lower()
    if "default_first_parent" in scope or "default_first_parent" in nm:
        return 2
    if "default_refs" in scope or "default_refs" in nm:
        return 1
    return 0

def _find_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lowmap = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lowmap:
            return lowmap[name.lower()]
    return None


# --------------------------
# Load MAIN metadata mapping
# --------------------------
def load_main_metadata(main_csv: Path) -> Tuple[Dict[str, Dict[str, str]], List[str]]:
    if not main_csv.exists():
        return {}, []

    df_main = pd.read_csv(main_csv, dtype=str, encoding="utf-8", keep_default_na=False)

    col_full = _find_col(df_main, ["full_name", "Full_name", "repo_full_name"])
    col_exec = _find_col(df_main, ["execution_environment", "Execution_Environment"])

    if not col_full:
        return {}, []

    df_main["_key"] = df_main[col_full].astype(str).str.lower().str.strip()

    # execution_environment as-is
    other_cols: List[str] = []
    for c in df_main.columns:
        cl = c.lower()
        if c in {"_key", col_full}:
            continue
        if col_exec and c == col_exec:
            continue
        if cl.startswith("api_"):
            continue
        other_cols.append(c)

    meta_map: Dict[str, Dict[str, str]] = {}
    for _, row in df_main.iterrows():
        k = str(row.get("_key", "")).strip()
        if not k:
            continue
        rec: Dict[str, str] = {}
        if col_exec and col_exec in df_main.columns:
            rec["execution_environment"] = str(row.get(col_exec, "") or "")
        for c in other_cols:
            rec[f"main_{c}"] = str(row.get(c, "") or "")
        meta_map[k] = rec

    meta_cols = []
    if col_exec:
        meta_cols.append("execution_environment")
    meta_cols.extend([f"main_{c}" for c in other_cols])
    return meta_map, meta_cols

META_MAP, META_COLS = load_main_metadata(MAIN_CSV)


# --------------------------
# Default-branch timeline filter (STRICT)
# --------------------------
def filter_timeline_to_default_branch(rec: Dict, default_scoped: bool) -> List[Dict]:
    """
    STRICT behavior:
      - If default_scoped: return timeline as-is (already mainline/default-scoped).
      - If NOT default_scoped:
          require evidence of on_default commits; otherwise return [] (do NOT accept repo-wide timelines).
    """
    timeline_rows = rec.get("timeline") or []
    if default_scoped:
        return timeline_rows

    events_dict = rec.get("events") or {}
    default_commits: Set[str] = set()

    for style in CANONICAL_STYLES:
        for ev in events_dict.get(style, []):
            if ev.get("on_default", 0) == 1:
                sha = (ev.get("commit") or "").strip()
                if sha:
                    default_commits.add(sha)

    if not default_commits:
        # STRICT: do not keep repo-wide timeline if we cannot prove default membership
        return []

    return [
        row for row in timeline_rows
        if (row.get("commit") or "").strip() in default_commits
    ]


# --------------------------
# Repo-level first/current status points
# --------------------------
def first_and_current_points(timeline_rows: List[Dict], cutoff_dt: datetime) -> Optional[Tuple[Dict, Dict]]:
    pts = []
    for r in (timeline_rows or []):
        dt = parse_iso(r.get("date"))
        if dt is None or dt > cutoff_dt:
            continue
        styles_now = normalize_styles(list(r.get("styles") or []))
        if not styles_now:
            continue
        sha = (r.get("commit") or "").strip()
        pts.append({"date": dt.astimezone(timezone.utc), "styles": styles_now, "commit": sha})

    if not pts:
        return None

    pts.sort(key=lambda x: x["date"])
    return pts[0], pts[-1]


# --------------------------
# Build episodes (tracks start/end commit SHAs)
# --------------------------
def build_episodes(timeline_rows: List[Dict], cutoff_dt: datetime) -> List[Dict]:
    rows = []
    for r in (timeline_rows or []):
        dt = parse_iso(r.get("date"))
        if dt is None or dt > cutoff_dt:
            continue
        styles_now = normalize_styles(list(r.get("styles") or []))
        if not styles_now:
            continue
        sha = (r.get("commit") or "").strip()
        rows.append({"date": dt, "styles": styles_now, "commit": sha})

    rows.sort(key=lambda r: r["date"])
    episodes: List[Dict] = []
    if not rows:
        return episodes

    cur_start = rows[0]["date"]
    cur_styles = rows[0]["styles"]
    cur_start_commit = rows[0]["commit"]
    prev_row = rows[0]

    for r in rows[1:]:
        dt = r["date"]
        styles_now = r["styles"]
        sha_now = r["commit"]

        if styles_now == cur_styles:
            prev_row = r
            continue

        end = min(dt, cutoff_dt)
        if end > cur_start:
            end_commit = sha_now or prev_row["commit"]
            episodes.append({
                "start": cur_start,
                "end": end,
                "styles": cur_styles,
                "duration_days": max(0.0, days_between(cur_start, end)),
                "start_commit": cur_start_commit,
                "end_commit": end_commit,
            })

        cur_start = dt
        cur_styles = styles_now
        cur_start_commit = sha_now
        prev_row = r

    if cur_start < cutoff_dt:
        episodes.append({
            "start": cur_start,
            "end": cutoff_dt,
            "styles": cur_styles,
            "duration_days": max(0.0, days_between(cur_start, cutoff_dt)),
            "start_commit": cur_start_commit,
            "end_commit": prev_row["commit"],
        })

    return episodes


# --------------------------
# Select ONE default-scoped JSON per repo (prefer default_first_parent)
# --------------------------
json_files = read_repo_jsons(OUTPUT_MINE)
if not json_files:
    raise FileNotFoundError(f"No miner JSON files found under: {OUTPUT_MINE}")

best_by_repo: Dict[str, Tuple[int, Path, Dict]] = {}  # repo -> (score, path, rec)
for jf in json_files:
    try:
        with jf.open("r", encoding="utf-8") as f:
            rec = json.load(f)
    except Exception:
        continue

    repo = rec.get("repo_name") or repo_from_filename(jf)
    if not repo:
        continue

    # STRICT: must be default-scoped
    if not is_default_scoped(jf, rec):
        continue

    score = default_scope_score(jf, rec)
    if score <= 0:
        continue

    if repo not in best_by_repo or score > best_by_repo[repo][0]:
        best_by_repo[repo] = (score, jf, rec)

selected = list(best_by_repo.values())
selected.sort(key=lambda t: str(t[1].name).lower())

print(f"[info] Found {len(json_files)} miner JSON files total.")
print(f"[info] Selected {len(selected)} DEFAULT-scoped repos (prefer default_first_parent).")

# --------------------------
# Main pipeline: build three outputs
# --------------------------
episode_rows: List[Dict] = []
boundary_event_rows: List[Dict] = []
repo_level_rows: List[Dict] = []

for score, jf, rec in selected:
    repo = rec.get("repo_name") or repo_from_filename(jf)

    # IMPORTANT: full_name should be Owner/Repo (matches GitHub + 5_Total_Repo.csv)
    full_name = (repo or "").replace("__", "/")
    key = full_name.lower().strip()

    meta = META_MAP.get(key, {})
    qa_issue = str(rec.get("qa_issue", "") or "")
    scope = str(rec.get("timeline_scope", "") or "")
    default_scoped = True  # selected guarantees this

    # ---- Episodes on default branch mainline only ----
    timeline = filter_timeline_to_default_branch(rec, default_scoped=default_scoped)
    episodes = build_episodes(timeline, CUTOFF_DT)

    # Keep episodes that are >=14d OR active at cutoff
    kept_eps = [
        ep for ep in episodes
        if (ep.get("duration_days", 0.0) >= PERSIST_DAYS) or (ep.get("end") == CUTOFF_DT)
    ]
    kept_eps.sort(key=lambda ep: ep["start"])
    for idx, ep in enumerate(kept_eps, start=1):
        ep["episode_index"] = idx

    # ---- Repo-level summary ----
    points = first_and_current_points(timeline, CUTOFF_DT)

    episodes_total = len(episodes)
    kept_episodes_total = len(kept_eps)

    first_state_label = first_styleset_str = first_date_utc = first_sha = ""
    current_state_label = current_styleset_str = current_date_utc = current_sha = ""
    current_since_utc = ""
    current_duration_days = ""

    if points:
        first_pt, last_pt = points

        first_date_utc = first_pt["date"].isoformat()
        first_sha = str(first_pt.get("commit") or "")
        first_styleset_str = styleset_to_str(set(first_pt.get("styles") or set()))
        first_state_label = label_from_set(set(first_pt.get("styles") or set()))

        current_date_utc = last_pt["date"].isoformat()
        current_sha = str(last_pt.get("commit") or "")
        current_styleset_str = styleset_to_str(set(last_pt.get("styles") or set()))
        current_state_label = label_from_set(set(last_pt.get("styles") or set()))

        if episodes_total > 0:
            last_ep = episodes[-1]
            current_since_utc = last_ep["start"].astimezone(timezone.utc).isoformat()
            current_duration_days = str(max(0.0, days_between(last_ep["start"], CUTOFF_DT)))
        else:
            current_since_utc = current_date_utc
            current_duration_days = str(max(0.0, days_between(last_pt["date"], CUTOFF_DT)))

    repo_row = {
        "repo_name": repo,
        "full_name": full_name,
        "cutoff_date": CUTOFF_ISO,
        "persist_days": PERSIST_DAYS,

        "timeline_scope": scope,
        "qa_issue": qa_issue,

        "episodes_total": episodes_total,
        "kept_episodes_total": kept_episodes_total,

        "first_status_date_utc": first_date_utc,
        "first_status_commit_sha": first_sha,
        "first_status_styleset": first_styleset_str,
        "first_status_label": first_state_label,

        "current_status_date_utc": current_date_utc,
        "current_status_commit_sha": current_sha,
        "current_status_styleset": current_styleset_str,
        "current_status_label": current_state_label,

        "current_status_since_utc": current_since_utc,
        "current_status_duration_days": current_duration_days,
    }
    repo_row.update(meta)
    repo_level_rows.append(repo_row)

    # ---- Episode timeline rows ----
    for ep in kept_eps:
        styleset = set(ep.get("styles") or set())
        styleset_str = styleset_to_str(styleset)
        row = {
            "repo_name": repo,
            "full_name": full_name,
            "cutoff_date": CUTOFF_ISO,
            "persist_days": PERSIST_DAYS,

            "timeline_scope": scope,
            "qa_issue": qa_issue,

            "episode_index": ep.get("episode_index", ""),
            "episode_start_utc": ep["start"].astimezone(timezone.utc).isoformat(),
            "episode_end_utc": ep["end"].astimezone(timezone.utc).isoformat(),
            "episode_duration_days": ep.get("duration_days", ""),

            "episode_styleset": styleset_str,
            "episode_state_label": label_from_set(styleset),

            "episode_start_commit_sha": str(ep.get("start_commit") or ""),
            "episode_end_commit_sha": str(ep.get("end_commit") or ""),

            "is_active_at_cutoff": 1 if ep.get("end") == CUTOFF_DT else 0,
            "is_persistent_ge14d": 1 if float(ep.get("duration_days", 0.0) or 0.0) >= PERSIST_DAYS else 0,
        }
        row.update(meta)
        episode_rows.append(row)

    # ---- Boundary indices by SHA (style, sha) -> [episode_index...] ----
    start_boundaries = defaultdict(list)
    end_boundaries = defaultdict(list)
    for ep in kept_eps:
        ssha = (ep.get("start_commit") or "").strip()
        esha = (ep.get("end_commit") or "").strip()
        for s in ep.get("styles") or []:
            if s in CANONICAL_STYLES:
                if ssha:
                    start_boundaries[(s, ssha)].append(ep["episode_index"])
                if esha:
                    end_boundaries[(s, esha)].append(ep["episode_index"])

    # ---- Build boundary events dataset ----
    events_dict = rec.get("events") or {}
    for style in CANONICAL_STYLES:
        for ev in events_dict.get(style, []):
            ev_type = str(ev.get("event", "") or "")
            raw_date = str(ev.get("date", "") or "")
            dt = parse_iso(raw_date)
            if dt is None:
                continue
            dt_utc = dt.astimezone(timezone.utc)
            sha = (ev.get("commit") or "").strip()

            # default-scoped: treat missing on_default as 1
            on_default_val = ev.get("on_default", 1)

            start_idxs = start_boundaries.get((style, sha), [])
            end_idxs = end_boundaries.get((style, sha), [])

            is_start_boundary = 1 if start_idxs else 0
            is_end_boundary = 1 if end_idxs else 0

            assigned_ep_idx = None
            if start_idxs:
                assigned_ep_idx = min(start_idxs)
            elif end_idxs:
                assigned_ep_idx = max(end_idxs)
            else:
                for ep in kept_eps:
                    if style in (ep.get("styles") or set()) and ep["start"] <= dt_utc < ep["end"]:
                        assigned_ep_idx = ep["episode_index"]
                        break

            in_kept = 1 if assigned_ep_idx is not None else 0

            # Keep only boundary added/removed commits
            if ev_type not in {"added", "removed"}:
                continue
            if not (is_start_boundary or is_end_boundary):
                continue
            if in_kept != 1:
                continue

            ep_start = ep_end = ""
            ep_dur = ""
            ep_styleset = ""
            ep_label = ""
            if assigned_ep_idx is not None:
                epx = next((x for x in kept_eps if x["episode_index"] == assigned_ep_idx), None)
                if epx:
                    ep_start = epx["start"].astimezone(timezone.utc).isoformat()
                    ep_end = epx["end"].astimezone(timezone.utc).isoformat()
                    ep_dur = epx.get("duration_days", "")
                    sset = set(epx.get("styles") or set())
                    ep_styleset = styleset_to_str(sset)
                    ep_label = label_from_set(sset)

            row = {
                "repo_name": repo,
                "full_name": full_name,
                "cutoff_date": CUTOFF_ISO,
                "persist_days": PERSIST_DAYS,

                "timeline_scope": scope,
                "qa_issue": qa_issue,

                "env_style": style,
                "event_type": ev_type,
                "event_date_raw": raw_date,
                "event_date_utc": dt_utc.isoformat(),
                "commit_sha": sha,
                "on_default": on_default_val,

                "episode_index": assigned_ep_idx if assigned_ep_idx is not None else "",
                "episode_start_utc": ep_start,
                "episode_end_utc": ep_end,
                "episode_duration_days": ep_dur,

                "episode_styleset": ep_styleset,
                "episode_state_label": ep_label,

                "is_episode_start_boundary": is_start_boundary,
                "is_episode_end_boundary": is_end_boundary,
            }
            row.update(meta)
            boundary_event_rows.append(row)

# --------------------------
# Write List_Boundary_Commit_Events.csv
# --------------------------
events_df = pd.DataFrame.from_records(boundary_event_rows)
if not events_df.empty:
    # keep default branch only
    events_df = events_df[events_df["on_default"].astype(str) == "1"].copy()

    sort_cols = ["repo_name", "episode_index", "event_date_utc", "env_style", "event_type"]
    sort_cols = [c for c in sort_cols if c in events_df.columns]
    if sort_cols:
        events_df.sort_values(sort_cols, inplace=True)

rq4_out_csv = OBS_OUTPUT / "List_Boundary_Commit_Events.csv"
events_df.to_csv(rq4_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote boundary-commit dataset: {rq4_out_csv}")

# --------------------------
# Write List_Change_Episodes.csv
# --------------------------
episodes_df = pd.DataFrame.from_records(episode_rows)
if not episodes_df.empty:
    sort_cols = ["repo_name", "episode_index", "episode_start_utc"]
    sort_cols = [c for c in sort_cols if c in episodes_df.columns]
    if sort_cols:
        episodes_df.sort_values(sort_cols, inplace=True)

episodes_out_csv = OBS_OUTPUT / "List_Change_Episodes.csv"
episodes_df.to_csv(episodes_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote episode timeline dataset: {episodes_out_csv}")

# --------------------------
# Write Repo_level_episodes.csv
# --------------------------
repo_df = pd.DataFrame.from_records(repo_level_rows)
if not repo_df.empty:
    sort_cols = ["repo_name"]
    sort_cols = [c for c in sort_cols if c in repo_df.columns]
    if sort_cols:
        repo_df.sort_values(sort_cols, inplace=True)

repo_out_csv = OBS_OUTPUT / "Repo_level_episodes.csv"
repo_df.to_csv(repo_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote repo-level first/current status: {repo_out_csv}")


[info] Found 440 miner JSON files total.
[info] Selected 440 DEFAULT-scoped repos (prefer default_first_parent).
[ok] Wrote boundary-commit dataset: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
[ok] Wrote episode timeline dataset: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Change_Episodes.csv
[ok] Wrote repo-level first/current status: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\Repo_level_episodes.csv
